---

## **DIPLOME UNIVERSITAIRE SDA**

## **ML Ops**

---

---
## **Prédiction de défaut de paiement (Loan Default)**

## **NB4_MLFLOW : Tracking des expériences avec MLflow**
---

### **Contexte**

Ce notebook fait suite au NB3_MODELISATION. Il reprend l'entraînement des modèles en ajoutant le tracking MLflow
pour logger les paramètres, métriques et le modèle retenu.

### **Prérequis**

Avant de lancer ce notebook, démarrer le serveur MLflow dans un terminal :
```
mlflow server --host 127.0.0.1 --port 8080
```

---

---

### Plan du notebook

| Section | Contenu |
|---------|--------|
| 1. Configuration | Imports, chemins, seed, MLflow client |
| 2. Chargement | Données préprocessées du NB2 |
| 3. Expérience MLflow | Création expérience, entraînement avec logging |
| 4. Ablation test | Run sans credit_lines_outstanding |
| 5. Sélection | Meilleur modèle, enregistrement MLflow |
| 6. Conclusion | Résultats, lien vers l'UI MLflow |

---

---

### Objectif du notebook

Ce notebook est un **livrable de pipeline (tracking MLflow)**. Il logge chaque expérience de modélisation pour assurer la traçabilité.

Il est conçu pour être :
- **reproductible** (chemins relatifs, seed fixé),
- **relançable** (chaque relance crée de nouveaux runs MLflow, ce qui permet de comparer les itérations),
- **traçable** (chaque run est enregistré dans MLflow),
- **orienté décisions** : le meilleur modèle est enregistré dans le registre MLflow.

---

In [1]:
# 1.1. Imports
from pathlib import Path
import sys
import os
import time
import pandas as pd
import numpy as np
import joblib
import mlflow
from mlflow import MlflowClient

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone
from sklearn.metrics import (
    f1_score, recall_score, precision_score, accuracy_score,
    roc_auc_score,
)

print(">> 1.1. Imports : OK")


>> 1.1. Imports : OK


In [2]:
# 1.2. Chemins relatifs (pipeline)
BASE = Path.cwd()
if not (BASE / "data").exists():
    BASE = BASE.parent
DATA_DIR = BASE / "data"
OUTPUT_DIR = BASE / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
MODEL_DIR = BASE / "model"
MODEL_DIR.mkdir(exist_ok=True)

print(f"Base    : {BASE}")
print(f"Data    : {DATA_DIR}")
print(">> 1.2. Chemins : OK")

# Timer global du notebook
notebook_start_time = time.time()


Base    : c:\STOCKAGE_XIA\DU SDA\MLOPS-01\Projet Diego
Data    : c:\STOCKAGE_XIA\DU SDA\MLOPS-01\Projet Diego\data
>> 1.2. Chemins : OK


In [3]:
# 1.3. Versions / seed
print("python  :", sys.version.split()[0])
print("pandas  :", pd.__version__)
print("numpy   :", np.__version__)
print("mlflow  :", mlflow.__version__)

SEED = 42
np.random.seed(SEED)

print(f"SEED    : {SEED}")
print(">> 1.3. Versions / seed : OK")

python  : 3.11.9
pandas  : 2.2.3
numpy   : 2.2.0
mlflow  : 2.19.0
SEED    : 42
>> 1.3. Versions / seed : OK


In [4]:
# 1.4. Constantes du projet (héritées du NB1_EDA)
CIBLE = "default"
COLONNE_A_SUPPRIMER = ["customer_id"]
FEATURES = [
    "credit_lines_outstanding",
    "loan_amt_outstanding",
    "total_debt_outstanding",
    "income",
    "years_employed",
    "fico_score",
    "ratio_dette_revenu",
    "ratio_dette_pret",
]
TEST_SIZE = 0.2

# Métrique primaire : Recall (contexte risque crédit)
METRIC_PRIMAIRE = "recall"
METRICS_SECONDAIRES = ["f1", "precision", "roc_auc"]

print(">> 1.4. Constantes projet : OK")


>> 1.4. Constantes projet : OK


In [5]:
# 1.5. Connexion au serveur MLflow
import os

MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://127.0.0.1:8080")
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

# Vérification de la connexion
try:
    experiments = client.search_experiments()
    print(f"Connexion MLflow OK sur {MLFLOW_TRACKING_URI}")
    print(f"  {len(experiments)} expérience(s) existante(s)")
except Exception as e:
    print(f"ERREUR : serveur MLflow non accessible sur {MLFLOW_TRACKING_URI}")
    print(f"Lancer d'abord : mlflow server --host 127.0.0.1 --port 8080")
    print(f"Ou définir la variable d'environnement MLFLOW_TRACKING_URI")
    raise e

print(">> 1.5. Connexion MLflow : OK")


Connexion MLflow OK sur http://127.0.0.1:8080
  5 expérience(s) existante(s)
>> 1.5. Connexion MLflow : OK


In [6]:
# 1.6. Création des 3 expériences (1 modèle = 1 expérience, conformément au TP)
EXPERIMENTS = {
    "Régression Logistique": {
        "name": "Loan_Default_LR",
        "tags": {
            "project_name": "loan-default-mlops",
            "model_type": "LogisticRegression",
            "dataset": "Loan_Data.csv (10 000 clients)",
            "team": "DU SDA",
            "mlflow.note.content": "Régression Logistique avec StandardScaler - class_weight balanced",
        },
    },
    "Decision Tree": {
        "name": "Loan_Default_DT",
        "tags": {
            "project_name": "loan-default-mlops",
            "model_type": "DecisionTreeClassifier",
            "dataset": "Loan_Data.csv (10 000 clients)",
            "team": "DU SDA",
            "mlflow.note.content": "Decision Tree - class_weight balanced, invariant aux échelles",
        },
    },
    "Random Forest": {
        "name": "Loan_Default_RF",
        "tags": {
            "project_name": "loan-default-mlops",
            "model_type": "RandomForestClassifier",
            "dataset": "Loan_Data.csv (10 000 clients)",
            "team": "DU SDA",
            "mlflow.note.content": "Random Forest 100 estimators - class_weight balanced",
        },
    },
}

# Créer les 3 expériences avec client.create_experiment (conformément au TP)
for model_key, exp_info in EXPERIMENTS.items():
    try:
        exp_id = client.create_experiment(name=exp_info["name"], tags=exp_info["tags"])
        print(f"  Expérience créée : {exp_info['name']} (id={exp_id})")
    except mlflow.exceptions.MlflowException:
        # L'expérience existe déjà (idempotent)
        exp = client.get_experiment_by_name(exp_info["name"])
        print(f"  Expérience existante : {exp_info['name']} (id={exp.experiment_id})")

print(">> 1.6. Expériences MLflow : OK")


  Expérience existante : Loan_Default_LR (id=879623651189863864)
  Expérience existante : Loan_Default_DT (id=218953985237684821)
  Expérience existante : Loan_Default_RF (id=913348496858838956)
>> 1.6. Expériences MLflow : OK


In [7]:
# 2.1. Chargement des données préprocessées (issues du NB2)
X_train = pd.read_csv(DATA_DIR / "X_train.csv")
X_test = pd.read_csv(DATA_DIR / "X_test.csv")
y_train = pd.read_csv(DATA_DIR / "y_train.csv").squeeze()
y_test = pd.read_csv(DATA_DIR / "y_test.csv").squeeze()

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"Features : {X_train.columns.tolist()}")
print(f"Ratio défaut train : {y_train.mean():.3f}")
print(f"Ratio défaut test  : {y_test.mean():.3f}")
print(">> 2.1. Chargement : OK")

X_train : (8000, 8)
X_test  : (2000, 8)
Features : ['credit_lines_outstanding', 'loan_amt_outstanding', 'total_debt_outstanding', 'income', 'years_employed', 'fico_score', 'ratio_dette_revenu', 'ratio_dette_pret']
Ratio défaut train : 0.185
Ratio défaut test  : 0.185
>> 2.1. Chargement : OK


In [8]:
# 2.1b. Quality gates sur les données du NB2
checks = {
    "Ratio train/test": (round(X_test.shape[0] / (X_train.shape[0] + X_test.shape[0]), 2),
                     abs(X_test.shape[0] / (X_train.shape[0] + X_test.shape[0]) - TEST_SIZE) < 0.01),
    "Features attendues": (len(X_train.columns), len(X_train.columns) == len(FEATURES)),
    "Features identiques": (X_train.columns.tolist() == X_test.columns.tolist(), True),
    "NaN X_train": (X_train.isnull().sum().sum(), X_train.isnull().sum().sum() == 0),
    "NaN X_test": (X_test.isnull().sum().sum(), X_test.isnull().sum().sum() == 0),
}

all_ok = True
for k, (valeur, condition) in checks.items():
    status = "[OK]" if condition else "[KO]"
    if not condition:
        all_ok = False
    print(f"  {status} {k}: {valeur}")

assert all_ok, "Quality gates KO - vérifier les sorties du NB2"
print(">> 2.1b. Quality gates : OK")


  [OK] Ratio train/test: 0.2
  [OK] Features attendues: 8
  [OK] Features identiques: True
  [OK] NaN X_train: 0
  [OK] NaN X_test: 0
>> 2.1b. Quality gates : OK


---

### 3. Entraînement avec tracking MLflow

Chaque modèle a sa propre expérience MLflow (1 modèle = 1 expérience).
Dans chaque expérience, 2 runs : baseline (toutes features) et ablation (sans credit_lines).

---


In [9]:
# 3.1. Définition des Pipelines
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# Normalisation des noms de run (sans accents, sans espaces)
import unicodedata
def normalize_run_name(name):
    # Supprime les accents et remplace les espaces
    name = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode("ascii")
    name = name.replace(" ", "_")
    return name


pipelines = [
    ("Régression Logistique", Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED)),
    ])),
    ("Decision Tree", Pipeline([
        ("clf", DecisionTreeClassifier(class_weight="balanced", random_state=SEED)),
    ])),
    ("Random Forest", Pipeline([
        ("clf", RandomForestClassifier(class_weight="balanced", n_estimators=100, random_state=SEED)),
    ])),
]

print(f"Modèles à entraîner : {[nom for nom, _ in pipelines]}")
print(">> 3.1. Pipelines définis : OK")

Modèles à entraîner : ['Régression Logistique', 'Decision Tree', 'Random Forest']
>> 3.1. Pipelines définis : OK


In [10]:
# 3.2. Entraînement avec logging MLflow (1 expérience par modèle)
resultats = []
# Sécurité : fermer tout run actif
mlflow.end_run()


for nom, pipe in pipelines:
    exp_name = EXPERIMENTS[nom]["name"]
    mlflow.set_experiment(exp_name)

    with mlflow.start_run(run_name=f"{normalize_run_name(nom)}_baseline") as run:
        # CV avant le fit
        cv_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="recall")

        # Entraînement (chrono sur le fit uniquement)
        t0 = time.time()
        pipe.fit(X_train, y_train)
        duree = round(time.time() - t0, 2)

        y_pred = pipe.predict(X_test)
        y_proba = pipe.predict_proba(X_test)[:, 1]

        metrics = {
            "recall_test": round(recall_score(y_test, y_pred), 4),
            "f1_test": round(f1_score(y_test, y_pred), 4),
            "precision_test": round(precision_score(y_test, y_pred), 4),
            "accuracy_test": round(accuracy_score(y_test, y_pred), 4),
            "roc_auc_test": round(roc_auc_score(y_test, y_proba), 4),
            "recall_cv_mean": round(cv_scores.mean(), 4),
            "recall_cv_std": round(cv_scores.std(), 4),
            "duree_fit_s": duree,
        }

        clf = pipe.named_steps['clf']
        params = clf.get_params()
        params_log = {k: str(v) for k, v in params.items() if v is not None}
        params_log["seed"] = str(SEED)
        params_log["features"] = str(X_train.columns.tolist())
        params_log["run_type"] = "baseline"

        mlflow.log_params(params_log)
        mlflow.log_metrics(metrics)
        mlflow.sklearn.log_model(
            sk_model=pipe,
            input_example=X_test.iloc[:1],
            artifact_path="model",
        )

        resultats.append({"modèle": nom, "run_type": "baseline", **metrics})
        print(f"{exp_name} | baseline : Recall={metrics['recall_test']} | F1={metrics['f1_test']} | {duree}s")

print("\n>> 3.2. Entraînement MLflow : OK")


c:\STOCKAGE_XIA\DU SDA\MLOPS-01\Projet Diego\venv\Lib\site-packages\mlflow\types\utils.py:435: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
c:\STOCKAGE_XIA\DU SDA\MLOPS-01\Projet Diego\venv\Lib\site-packages\mlflow\types\utils.py:435: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If 

Loan_Default_LR | baseline : Recall=1.0 | F1=0.9906 | 0.01s
🏃 View run Regression_Logistique_baseline at: http://127.0.0.1:8080/#/experiments/879623651189863864/runs/b01ea3390a274da9b1e3366ff4976d52
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/879623651189863864


c:\STOCKAGE_XIA\DU SDA\MLOPS-01\Projet Diego\venv\Lib\site-packages\mlflow\types\utils.py:435: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
c:\STOCKAGE_XIA\DU SDA\MLOPS-01\Projet Diego\venv\Lib\site-packages\mlflow\types\utils.py:435: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If 

Loan_Default_DT | baseline : Recall=0.9676 | F1=0.9689 | 0.02s
🏃 View run Decision_Tree_baseline at: http://127.0.0.1:8080/#/experiments/218953985237684821/runs/8bde47c21f474e2496da43a3a3b4140b
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/218953985237684821


c:\STOCKAGE_XIA\DU SDA\MLOPS-01\Projet Diego\venv\Lib\site-packages\mlflow\types\utils.py:435: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
c:\STOCKAGE_XIA\DU SDA\MLOPS-01\Projet Diego\venv\Lib\site-packages\mlflow\types\utils.py:435: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If 

Loan_Default_RF | baseline : Recall=0.9568 | F1=0.9752 | 0.36s
🏃 View run Random_Forest_baseline at: http://127.0.0.1:8080/#/experiments/913348496858838956/runs/7cdd826de2864fd48b3e8ae1bab49fd9
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/913348496858838956

>> 3.2. Entraînement MLflow : OK


In [11]:
# 3.3. Tableau comparatif (baseline uniquement)
# Ce premier tableau compare les 3 modèles sur toutes les features.
# Il permet de choisir le meilleur modèle AVANT de tester la robustesse.
df_resultats = pd.DataFrame(resultats)
print("=" * 80)
print("COMPARAISON DES MODÈLES (baseline - toutes features)")
print("=" * 80)
print(df_resultats.to_string(index=False))


COMPARAISON DES MODÈLES (baseline - toutes features)
               modèle run_type  recall_test  f1_test  precision_test  accuracy_test  roc_auc_test  recall_cv_mean  recall_cv_std  duree_fit_s
Régression Logistique baseline       1.0000   0.9906          0.9814         0.9965        1.0000          1.0000         0.0000         0.01
        Decision Tree baseline       0.9676   0.9689          0.9702         0.9885        0.9804          0.9595         0.0155         0.02
        Random Forest baseline       0.9568   0.9752          0.9944         0.9910        0.9997          0.9716         0.0055         0.36


---

### 4. Ablation test - sans credit_lines_outstanding

Chaque modèle est ré-entraîné sans `credit_lines_outstanding` dans sa propre expérience.

---


In [12]:
# 4.1. Ablation test (loggé dans l'expérience de chaque modèle)
FEATURES_SANS_CL = [f for f in X_train.columns if f != "credit_lines_outstanding"]
# Sécurité : fermer tout run actif
mlflow.end_run()


for nom, pipe in pipelines:
    exp_name = EXPERIMENTS[nom]["name"]
    mlflow.set_experiment(exp_name)
    pipe_clone = clone(pipe)

    with mlflow.start_run(run_name=f"{normalize_run_name(nom)}_sans_credit_lines") as run:
        # CV avant le fit (même ordre que baseline)
        cv_scores = cross_val_score(pipe_clone, X_train[FEATURES_SANS_CL], y_train, cv=cv, scoring="recall")

        # Entraînement (chrono sur le fit uniquement)
        t0 = time.time()
        pipe_clone.fit(X_train[FEATURES_SANS_CL], y_train)
        duree = round(time.time() - t0, 2)

        y_pred = pipe_clone.predict(X_test[FEATURES_SANS_CL])
        y_proba = pipe_clone.predict_proba(X_test[FEATURES_SANS_CL])[:, 1]

        metrics = {
            "recall_test": round(recall_score(y_test, y_pred), 4),
            "f1_test": round(f1_score(y_test, y_pred), 4),
            "precision_test": round(precision_score(y_test, y_pred), 4),
            "accuracy_test": round(accuracy_score(y_test, y_pred), 4),
            "roc_auc_test": round(roc_auc_score(y_test, y_proba), 4),
            "recall_cv_mean": round(cv_scores.mean(), 4),
            "recall_cv_std": round(cv_scores.std(), 4),
            "duree_fit_s": duree,
        }

        clf = pipe_clone.named_steps['clf']
        params = clf.get_params()
        params_log = {k: str(v) for k, v in params.items() if v is not None}
        params_log["seed"] = str(SEED)
        params_log["features"] = str(FEATURES_SANS_CL)
        params_log["run_type"] = "ablation_sans_credit_lines"

        mlflow.log_params(params_log)
        mlflow.log_metrics(metrics)
        mlflow.sklearn.log_model(
            sk_model=pipe_clone,
            input_example=X_test[FEATURES_SANS_CL].iloc[:1],
            artifact_path="model",
        )

        resultats.append({"modèle": nom, "run_type": "ablation_sans_credit_lines", **metrics})
        print(f"{exp_name} | ablation : Recall={metrics['recall_test']} | F1={metrics['f1_test']} | {duree}s")

# Tableau complet : compare baseline et ablation côte à côte.
# Permet de mesurer l'impact de credit_lines_outstanding sur chaque modèle.
df_resultats = pd.DataFrame(resultats)
print("\n" + "=" * 80)
print(f"COMPARAISON COMPLÈTE ({len(df_resultats)} runs : baseline + ablation)")
print("=" * 80)
print(df_resultats.to_string(index=False))
print("\n>> 4.1. Ablation test MLflow : OK")


c:\STOCKAGE_XIA\DU SDA\MLOPS-01\Projet Diego\venv\Lib\site-packages\mlflow\types\utils.py:435: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
c:\STOCKAGE_XIA\DU SDA\MLOPS-01\Projet Diego\venv\Lib\site-packages\mlflow\types\utils.py:435: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If 

Loan_Default_LR | ablation : Recall=0.9838 | F1=0.9467 | 0.01s
🏃 View run Regression_Logistique_sans_credit_lines at: http://127.0.0.1:8080/#/experiments/879623651189863864/runs/e9700ab07735437d90a30f317e148e73
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/879623651189863864


c:\STOCKAGE_XIA\DU SDA\MLOPS-01\Projet Diego\venv\Lib\site-packages\mlflow\types\utils.py:435: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
c:\STOCKAGE_XIA\DU SDA\MLOPS-01\Projet Diego\venv\Lib\site-packages\mlflow\types\utils.py:435: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If 

Loan_Default_DT | ablation : Recall=0.9297 | F1=0.9361 | 0.02s
🏃 View run Decision_Tree_sans_credit_lines at: http://127.0.0.1:8080/#/experiments/218953985237684821/runs/160dc79ed00e42df8268287e0a2f39f7
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/218953985237684821


c:\STOCKAGE_XIA\DU SDA\MLOPS-01\Projet Diego\venv\Lib\site-packages\mlflow\types\utils.py:435: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
c:\STOCKAGE_XIA\DU SDA\MLOPS-01\Projet Diego\venv\Lib\site-packages\mlflow\types\utils.py:435: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If 

Loan_Default_RF | ablation : Recall=0.9459 | F1=0.9537 | 0.44s
🏃 View run Random_Forest_sans_credit_lines at: http://127.0.0.1:8080/#/experiments/913348496858838956/runs/41b5e0679a674bfca68658ae79a78cf7
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/913348496858838956

COMPARAISON COMPLÈTE (6 runs : baseline + ablation)
               modèle                   run_type  recall_test  f1_test  precision_test  accuracy_test  roc_auc_test  recall_cv_mean  recall_cv_std  duree_fit_s
Régression Logistique                   baseline       1.0000   0.9906          0.9814         0.9965        1.0000          1.0000         0.0000         0.01
        Decision Tree                   baseline       0.9676   0.9689          0.9702         0.9885        0.9804          0.9595         0.0155         0.02
        Random Forest                   baseline       0.9568   0.9752          0.9944         0.9910        0.9997          0.9716         0.0055         0.36
Régression Logistique ablat

---

### 5. Sélection et enregistrement du meilleur modèle

---


In [13]:
# 5.1. Sélection du meilleur modèle (critère : Recall)
df_baseline = df_resultats[df_resultats["run_type"] == "baseline"]
# Tri par recall puis F1 en cas d'égalité
meilleur_idx = df_baseline.sort_values(["recall_test", "f1_test"], ascending=False).index[0]
meilleur_nom = df_baseline.loc[meilleur_idx, "modèle"]
meilleur_recall = df_baseline.loc[meilleur_idx, "recall_test"]

# Récupérer le pipeline par son nom (plus robuste que par index)
meilleur_pipe = next(pipe for nom, pipe in pipelines if nom == meilleur_nom)

print(f"Meilleur modèle : {meilleur_nom}")
print(f"Recall test     : {meilleur_recall}")
print()

# Sauvegarder le meilleur modèle en local
joblib.dump(meilleur_pipe, MODEL_DIR / "best_model.joblib")
print(f"Modèle sauvegardé : {MODEL_DIR / 'best_model.joblib'}")
print(">> 5.1. Sélection : OK")

# Sauvegarde des 3 modèles (pour app multimodèle)
for nom, pipe in pipelines:
    nom_fichier = nom.lower().replace(' ', '_').replace('é', 'e')
    joblib.dump(pipe, MODEL_DIR / f"model_{nom_fichier}.joblib")
    print(f"Modèle sauvegardé : model_{nom_fichier}.joblib")

# Sauvegarde du meta JSON (cohérent avec MLflow)
import json as json_lib
meta = {
    "modèle": meilleur_nom,
    "recall_test": float(meilleur_recall),
    "f1_test": float(df_baseline.loc[meilleur_idx, "f1_test"]),
    "seed": SEED,
    "date": pd.Timestamp.now().isoformat(),
}
meta_path = MODEL_DIR / "best_model_meta.json"
with open(meta_path, "w") as f:
    json_lib.dump(meta, f, indent=2)
print(f"Meta sauvegardée : {meta_path}")

# Sauvegarde du tableau comparatif en CSV
csv_path = OUTPUT_DIR / "NB4_comparaison_modeles.csv"
df_resultats.to_csv(csv_path, index=False)
print(f"Tableau comparatif sauvegardé : {csv_path}")


Meilleur modèle : Régression Logistique
Recall test     : 1.0

Modèle sauvegardé : c:\STOCKAGE_XIA\DU SDA\MLOPS-01\Projet Diego\model\best_model.joblib
>> 5.1. Sélection : OK
Modèle sauvegardé : model_regression_logistique.joblib
Modèle sauvegardé : model_decision_tree.joblib
Modèle sauvegardé : model_random_forest.joblib
Meta sauvegardée : c:\STOCKAGE_XIA\DU SDA\MLOPS-01\Projet Diego\model\best_model_meta.json
Tableau comparatif sauvegardé : c:\STOCKAGE_XIA\DU SDA\MLOPS-01\Projet Diego\outputs\NB4_comparaison_modeles.csv


In [14]:
# 5.2. Enregistrement dans le registre MLflow
best_exp_name = EXPERIMENTS[meilleur_nom]["name"]
best_exp = mlflow.set_experiment(best_exp_name)

# Le run name utilise des underscores au lieu des espaces
run_name_filtre = normalize_run_name(meilleur_nom) + "_baseline"

runs = client.search_runs(
    experiment_ids=[best_exp.experiment_id],
    filter_string=f"tags.mlflow.runName = '{run_name_filtre}'",
    order_by=["metrics.recall_test DESC", "attributes.start_time DESC"],
    max_results=1,
)

if runs:
    best_run = runs[0]
    model_uri = f"runs:/{best_run.info.run_id}/model"

    result = mlflow.register_model(model_uri=model_uri, name="best_loan_default_model")

    # Logger le tableau comparatif comme artifact du meilleur modèle
    csv_path = OUTPUT_DIR / "NB4_comparaison_modeles.csv"
    if csv_path.exists():
        mlflow.log_artifact(str(csv_path))
        print(f"Tableau comparatif loggé comme artifact")
    print(f"Modèle enregistré dans le registre MLflow : {result.name} (version {result.version})")
else:
    print("Run non trouvé - vérifier le nom du run")

print(">> 5.2. Enregistrement MLflow : OK")


Registered model 'best_loan_default_model' already exists. Creating a new version of this model...
2026/04/06 12:25:50 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: best_loan_default_model, version 16


Tableau comparatif loggé comme artifact
Modèle enregistré dans le registre MLflow : best_loan_default_model (version 16)
>> 5.2. Enregistrement MLflow : OK


Created version '16' of model 'best_loan_default_model'.


In [15]:
print("=== Temps total d'exécution du notebook ===")
# Temps total d'exécution du notebook
notebook_total_time = time.time() - notebook_start_time
print(f"Temps total du notebook : {notebook_total_time:.1f}s ({notebook_total_time/60:.1f} min)")


=== Temps total d'exécution du notebook ===
Temps total du notebook : 19.2s (0.3 min)


---
## Conclusion NB4 - Tracking MLflow
---

### Ce qui a été fait dans ce notebook :

| Constat | Preuve | Décision |
|---------|--------|----------|
| 3 expériences MLflow créées (1 par modèle) | UI MLflow : Loan_Default_LR, Loan_Default_DT, Loan_Default_RF | Conformité avec la consigne 1 modèle = 1 expérience |
| 6 runs loggés (3 baseline + 3 ablation) | Paramètres, métriques et modèle tracés | Traçabilité complète des expériences |
| Le meilleur modèle est enregistré dans le registre MLflow | Model Registry | Modèle prêt pour le déploiement |

### Pour visualiser les résultats :

Ouvrir dans le navigateur : **http://127.0.0.1:8080**

### Choix retenus :

- Le meilleur modèle (sélectionné par Recall) est disponible dans le registre MLflow
- Il est aussi sauvegardé en local dans `model/best_model.joblib` pour l'app Streamlit
